In [3]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import types
from pyspark.sql import functions as F
import os
import glob

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .config("spark.ui.port", "4040") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

25/03/06 13:11:12 WARN Utils: Your hostname, DESKTOP-QRKC0LG resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/06 13:11:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/06 13:11:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/06 13:11:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
# Question 1
print(spark.version)

3.5.5


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-06 12:00:18--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.33.163.101, 13.33.163.58, 13.33.163.188, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.163.101|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  16.1MB/s    in 4.0s    

2025-03-06 12:00:22 (15.5 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [7]:
df = spark.read.parquet('/home/lpop22/LKzoomcamp2025/05-batch-processing/data/yellow_tripdata_2024-10.parquet')

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/lpop22/LKzoomcamp2025/05-batch-processing/data/yellow_tripdata_2024-10.parquet.

In [14]:
df = df.repartition(4)

In [22]:
df.write.parquet('homework/taxidata/')

In [23]:
# Question 2
!ls -lh homework/taxidata/*.parquet


-rw-r--r-- 1 lpop22 lpop22 23M Mar  6 12:11 homework/taxidata/part-00000-d5713553-e254-4d7b-8e2c-ddd78b069e74-c000.snappy.parquet
-rw-r--r-- 1 lpop22 lpop22 23M Mar  6 12:11 homework/taxidata/part-00001-d5713553-e254-4d7b-8e2c-ddd78b069e74-c000.snappy.parquet
-rw-r--r-- 1 lpop22 lpop22 23M Mar  6 12:11 homework/taxidata/part-00002-d5713553-e254-4d7b-8e2c-ddd78b069e74-c000.snappy.parquet
-rw-r--r-- 1 lpop22 lpop22 23M Mar  6 12:11 homework/taxidata/part-00003-d5713553-e254-4d7b-8e2c-ddd78b069e74-c000.snappy.parquet


In [24]:
df.head()

Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 10, 3, 3, 40, 19), tpep_dropoff_datetime=datetime.datetime(2024, 10, 3, 3, 46, 11), passenger_count=1, trip_distance=0.6, RatecodeID=1, store_and_fwd_flag='N', PULocationID=48, DOLocationID=161, payment_type=1, fare_amount=6.5, extra=3.5, mta_tax=0.5, tip_amount=2.9, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=14.4, congestion_surcharge=2.5, Airport_fee=0.0)

In [25]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [1]:
df.dropDuplicates()
df.count()

NameError: name 'df' is not defined

In [32]:
# Question 3
df_filtered = df.filter(
    (F.col("tpep_pickup_datetime")>= F.lit("2024-10-15")) & 
    (F.col("tpep_pickup_datetime") < F.lit("2024-10-16"))
)

In [33]:
df_filtered.count()

128893

In [44]:
df = df.withColumn("trip_duration", 
                   (F.unix_timestamp(F.col("tpep_dropoff_datetime")) - F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 60 / 60
                  )

In [40]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- trip_duration: double (nullable = true)



In [45]:
# Question 4
df.agg(F.max(F.col("trip_duration"))).show()

+------------------+
|max(trip_duration)|
+------------------+
| 162.6177777777778|
+------------------+



In [ ]:
# Question 5
# Spark UI runs on port 4040 by default if not in use by something else

In [46]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-06 12:43:06--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.33.163.13, 13.33.163.101, 13.33.163.188, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.163.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-06 12:43:06 (1.46 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [48]:
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

In [49]:
df_zones.head()

Row(LocationID=1, Borough='EWR', Zone='Newark Airport', service_zone='EWR')

In [52]:
df_joined = df.join(df_zones, df["PULocationID"] == df_zones["LocationID"], how="left")

In [54]:
df_joined.head()

Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 10, 3, 3, 40, 19), tpep_dropoff_datetime=datetime.datetime(2024, 10, 3, 3, 46, 11), passenger_count=1, trip_distance=0.6, RatecodeID=1, store_and_fwd_flag='N', PULocationID=48, DOLocationID=161, payment_type=1, fare_amount=6.5, extra=3.5, mta_tax=0.5, tip_amount=2.9, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=14.4, congestion_surcharge=2.5, Airport_fee=0.0, trip_duration=0.09777777777777777, LocationID=48, Borough='Manhattan', Zone='Clinton East', service_zone='Yellow Zone')

In [55]:
df_zones_grouped = df_joined.groupBy(F.col("Zone")).count().withColumnRenamed("count", "trip_count")

In [57]:
# Question 6
df_zones_grouped.orderBy(F.col("trip_count"), ascending=True).show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|       Rikers Island|         2|
|       Arden Heights|         2|
|         Jamaica Bay|         3|
| Green-Wood Cemetery|         3|
|Charleston/Totten...|         4|
|   Rossville/Woodrow|         4|
|       Port Richmond|         4|
|Eltingville/Annad...|         4|
|       West Brighton|         4|
|         Great Kills|         6|
|        Crotona Park|         6|
|Heartland Village...|         7|
|     Mariners Harbor|         7|
|Saint George/New ...|         9|
|             Oakwood|         9|
|       Broad Channel|        10|
|New Dorp/Midland ...|        10|
|         Westerleigh|        12|
|     Pelham Bay Park|        12|
+--------------------+----------+
only showing top 20 rows

